In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id
from pyspark.sql.types import StringType, IntegerType, DoubleType, TimestampType

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("table", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table")

In [0]:
df_bronze = spark.read.table(f"{catalog}.{schema}.{table}")


In [0]:
df_bronze_cleaned = df_bronze.select(
    col("vendor_name").alias("ct_vendor_name").cast(StringType()),
    col("Trip_Pickup_DateTime").alias("ts_trip_pickup").cast(TimestampType()),
    col("Trip_Dropoff_DateTime").alias("ts_trip_dropoff").cast(TimestampType()),
    col("Passenger_Count").alias("passager_count").cast(IntegerType()),
    col("Trip_Distance").alias("trip_distance").cast(DoubleType()),
    col("Start_Lon").alias("start_lon").cast(StringType()),
    col("Start_Lat").alias("start_lat").cast(StringType()),
    col("End_Lon").alias("end_lon").cast(StringType()),
    col("End_Lat").alias("end_lat").cast(StringType()),
    col("Payment_Type").alias("ct_payment_type").cast(StringType()),
    col("Total_Amt").alias("total_amount").cast(DoubleType())
)

In [0]:
df_bronze_enriched = (df_bronze_cleaned
    .withColumns({
        "pk_trip": monotonically_increasing_id(),
        "trip_duration_in_min": col("ts_trip_dropoff").cast("long") - col("ts_trip_pickup").cast("long")/60
    }))

In [0]:
df_bronze_enriched.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_trip_data")